# Estados de carga unitarios por nodo de piping y rigidez por dirección

Un notebook para las dos etapas:

**Etapa 1 (siempre):** adjunta el `.s2k` base de SAP2000 (kgf-m, con los grupos de nodos de piping asignados).
Por cada grupo se crea un **estado de carga** con su mismo nombre, con 1 kgf en X y Y, y 1 kgf en Z hacia abajo (negativa), sobre el joint del grupo.
Se genera `<modelo>_NODOS.s2k`. Descárgalo y córrelo en SAP. Una salida `_NODOS` también puede volver a entrar.

**Etapa 2 (cuando tengas resultados):** adjunta además el Excel/CSV de la tabla **Joint Displacements** de SAP
(columnas `Joint`, `OutputCase`, `U1`, `U2`, `U3`), con un caso de salida por nodo, llamado igual que el nodo.
Se calcula **k_i = F_i / U_i** en cada dirección y se exporta `rigidez_por_nodo.csv` y `.xlsx`.

El notebook no crea ni renombra grupos. Cada nodo de piping es un grupo con un único joint: los grupos `ALL`, `none` y cualquier grupo con más de un joint se saltan y se listan. Los nodos de la lista sin grupo también se saltan.

Con la carga en las tres direcciones a la vez, k_i es la **rigidez aparente**: exacta si no hay acoplamiento
entre direcciones, aproximada si lo hay.

In [ ]:
import re
import glob
from pathlib import Path

import numpy as np
import pandas as pd

INPUT_DIR = "/kaggle/input"
OUTPUT_DIR = "/kaggle/working"

MODEL_FILE = None            # .s2k base; None = detectar automáticamente
NODES_FILE = None            # opcional: Excel/CSV con la lista de nodos de piping (columna A). None = todos los grupos
DESPLAZAMIENTOS_FILE = None  # opcional: Excel/CSV de Joint Displacements. None = detectar; si no hay, solo etapa 1

FORCE_KGF = (1.0, 1.0, -1.0)  # fuerza en kgf en X, Y, Z a cada estado; Z negativa = hacia abajo
LOAD_CS = "GLOBAL"           # sistema de coordenadas de las cargas
ENCODING = "latin-1"         # preserva los bytes del .s2k al reescribirlo
IGNORAR_GRUPOS = {"all", "none"}  # grupos de SAP que nunca son nodos de piping (sin distinguir mayúsculas)

In [ ]:
def find_files(exts, must_contain=None):
    hits = sorted(p for p in glob.glob(f"{INPUT_DIR}/**/*", recursive=True)
                  if Path(p).suffix.lower() in exts
                  and (must_contain is None or any(s in Path(p).name.lower() for s in must_contain)))
    return hits


model_hits = find_files({".s2k"})
model_path = MODEL_FILE or (model_hits[0] if len(model_hits) == 1 else None)
if model_path is None:
    raise FileNotFoundError(f"Se esperaba un único .s2k en {INPUT_DIR}, encontrados: {model_hits}. Fija MODEL_FILE.")
# Si el .s2k ya es una salida de este notebook (_NODOS), se regenera igual: no duplica estados, casos ni cargas.

nodes_path = NODES_FILE or None
if nodes_path is None:
    nodes_hits = [p for p in find_files({".xlsx", ".xls", ".csv"}) if "nodo" in Path(p).name.lower()]
    nodes_path = nodes_hits[0] if len(nodes_hits) == 1 else None

desp_path = DESPLAZAMIENTOS_FILE
if desp_path is None:
    desp_hits = [p for p in find_files({".xlsx", ".xls", ".csv"}, ["desplaz", "displac", "joint"])
                 if p != nodes_path]
    desp_path = desp_hits[0] if len(desp_hits) == 1 else None

print("Modelo:", model_path)
print("Lista de nodos:", nodes_path or "(no se usa; se toman todos los grupos de nodos)")
print("Desplazamientos SAP:", desp_path or "(no hay: solo se genera el .s2k)")

In [ ]:
def normalize_name(v):
    s = str(v).strip()
    return s[:-2] if re.fullmatch(r"\d+\.0", s) else s


def read_table(path):
    if path.lower().endswith(".csv"):
        return pd.read_csv(path, header=None, dtype=str)
    return pd.read_excel(path, header=None, dtype=str)


def read_nodes(path):
    nodes = read_table(path).iloc[:, 0].dropna().map(normalize_name)
    if nodes.iloc[0].lower() in {"nodo", "nodos", "node"}:
        nodes = nodes.iloc[1:]
    dup = nodes[nodes.duplicated(keep=False)]
    if len(dup):
        raise ValueError(f"Nodos duplicados en la lista:\n{sorted(set(dup))}")
    return list(nodes)


nodes_list = read_nodes(nodes_path) if nodes_path else None
if nodes_list is not None:
    print(f"Nodos en la lista: {len(nodes_list)}")

In [ ]:
TABLE_RE = re.compile(r'^TABLE:\s+"(.+?)"')
KV_RE = re.compile(r'(\w+)=("[^"]*"|\S+)')


def parse_s2k(path):
    with open(path, encoding=ENCODING) as f:
        lines = f.read().splitlines()
    tables = {}
    current = None
    for line in lines:
        m = TABLE_RE.match(line.strip())
        if m:
            current = m.group(1)
            tables.setdefault(current, [])
        elif current is not None and "=" in line:
            tables[current].append({k: v.strip('"') for k, v in KV_RE.findall(line)})
    return lines, tables


def group_members(tables):
    """{grupo: {joints}} a partir de la tabla de asignaciones de grupos."""
    members = {}
    for name, rows in tables.items():
        if "GROUP" not in name.upper() or "ASSIGN" not in name.upper():
            continue
        for row in rows:
            if row.get("ObjectType", "").lower() == "joint":
                members.setdefault(row["GroupName"], set()).add(row["ObjectLabel"])
    return members


lines, tables = parse_s2k(model_path)

units = next((r["CurrUnits"] for r in tables.get("PROGRAM CONTROL", []) if "CurrUnits" in r), None)
if units is None:
    print("Aviso: no encontré CurrUnits en PROGRAM CONTROL; verifica a mano que el modelo esté en kgf-m.")
else:
    parts = [p.strip().lower() for p in units.split(",")]
    if parts[:2] != ["kgf", "m"]:
        raise ValueError(f"El modelo está en '{units}', no en kgf-m. Cambia unidades en SAP.")
    print("Unidades del modelo:", units)

members = group_members(tables)
defined_patterns = {r["LoadPat"] for r in tables.get("LOAD PATTERN DEFINITIONS", []) if "LoadPat" in r}
defined_cases = {r["Case"] for r in tables.get("LOAD CASE DEFINITIONS", []) if "Case" in r}
assigned = {(r["Case"], r["LoadName"]) for r in tables.get("CASE - STATIC 1 - LOAD ASSIGNMENTS", [])
            if r.get("LoadType", "").lower() == "load pattern"}
print(f"Grupos con joints: {len(members)}. Estados definidos: {len(defined_patterns)}")

In [ ]:
# Mapeo: el nombre del grupo es el nodo de piping; su único joint es el punto de carga en SAP.
if nodes_list is not None:
    candidatos = nodes_list
else:
    candidatos = [g for g in members if g.lower() not in IGNORAR_GRUPOS]
mapping, sin_grupo, otros = {}, [], []
for node in candidatos:
    if node not in members:
        sin_grupo.append(node)
        continue
    js = sorted(members[node])
    if len(js) != 1:
        # Un nodo de piping tiene un único joint. Con lista de nodos, es un error; sin lista, es otro grupo de SAP.
        if nodes_list is not None:
            raise ValueError(f"Nodo '{node}': el grupo tiene {len(js)} joints {js}; no se sabe a cuál aplicar la carga")
        otros.append(f"{node} ({len(js)} joints)")
        continue
    mapping[node] = js[0]

if otros:
    print("Grupos con más de un joint, no son nodos de piping (se saltan):")
    print("  " + ", ".join(otros))
if not mapping:
    raise ValueError("Ningún nodo tiene grupo en el .s2k. Revisa los nombres de grupo.")

mapeo = pd.DataFrame([{"nodo_piping": n, "joint_sap": j} for n, j in mapping.items()],
                     columns=["nodo_piping", "joint_sap"])
print(f"Nodos con grupo y joint: {len(mapping)}. Sin grupo (se saltan): {len(sin_grupo)}")
if sin_grupo:
    print("  " + ", ".join(sin_grupo))
mapeo

In [ ]:
def fmt(v):
    return f'"{v}"'


def num(v):
    return f"{float(v):.10g}"


def add_rows(lines, table_name, new_rows):
    """Inserta filas justo debajo de la cabecera de la tabla; si no existe, la crea antes de END TABLE DATA."""
    if not new_rows:
        return lines
    for i, line in enumerate(lines):
        m = TABLE_RE.match(line.strip())
        if m and m.group(1) == table_name:
            return lines[:i + 1] + new_rows + lines[i + 1:]
    block = ["", f'TABLE:  "{table_name}"'] + new_rows
    end = next((i for i, l in enumerate(lines) if l.strip() == "END TABLE DATA"), len(lines))
    return lines[:end] + block + lines[end:]


def replace_existing(lines, pairs):
    """Quita las cargas de joint ya existentes para los pares (joint, estado) de este run."""
    keep, replaced, nonzero, table = [], 0, 0, None
    for line in lines:
        m = TABLE_RE.match(line.strip())
        if m:
            table = m.group(1)
        elif table == "JOINT LOADS - FORCE" and "=" in line:
            row = {k: v.strip('"') for k, v in KV_RE.findall(line)}
            if (row.get("Joint"), row.get("LoadPat")) in pairs:
                replaced += 1
                if any(float(row.get(k, 0)) != 0 for k in ["F1", "F2", "F3", "M1", "M2", "M3"]):
                    nonzero += 1
                continue
        keep.append(line)
    print(f"Cargas existentes reemplazadas: {replaced} (de ellas {nonzero} distintas de cero)")
    return keep


fx, fy, fz = (num(v) for v in FORCE_KGF)
new_patterns = [n for n in mapping if n not in defined_patterns]
pattern_rows = [f"   LoadPat={fmt(p)}  DesignType=Other  SelfWtMult=0" for p in new_patterns]
joint_rows = [
    f"   Joint={fmt(mapping[n])}  LoadPat={fmt(n)}  CoordSys={LOAD_CS}  "
    f"F1={fx}  F2={fy}  F3={fz}  M1=0  M2=0  M3=0"
    for n in mapping
]

# Un caso lineal estático por nodo, con su load pattern asignado: así SAP lo resuelve y exporta sus desplazamientos.
case_rows = [f'   Case={fmt(n)}  Type=LinStatic  InitialCond=Zero' for n in mapping if n not in defined_cases]
assign_rows = [f'   Case={fmt(n)}  LoadType="Load pattern"  LoadName={fmt(n)}  LoadSF=1'
               for n in mapping if (n, n) not in assigned]
print(f"Casos de carga nuevos: {len(case_rows)}. Asignaciones nuevas: {len(assign_rows)}")

new_lines = replace_existing(lines, {(mapping[n], n) for n in mapping})
new_lines = add_rows(new_lines, "LOAD PATTERN DEFINITIONS", pattern_rows)
new_lines = add_rows(new_lines, "LOAD CASE DEFINITIONS", case_rows)
new_lines = add_rows(new_lines, "CASE - STATIC 1 - LOAD ASSIGNMENTS", assign_rows)
new_lines = add_rows(new_lines, "JOINT LOADS - FORCE", joint_rows)

out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
stem = Path(model_path).stem
out_s2k = out_dir / (f"{stem}.s2k" if stem.endswith("_NODOS") else f"{stem}_NODOS.s2k")
with open(out_s2k, "w", encoding=ENCODING, newline="\n") as f:
    f.write("\n".join(new_lines) + "\n")
mapeo.to_csv(out_dir / "mapeo_nodos.csv", index=False)
print(f"Escrito: {out_s2k}")
print(f"{len(joint_rows)} estados con carga de {FORCE_KGF} kgf ({len(pattern_rows)} nuevos)")

In [ ]:
_, out_tables = parse_s2k(out_s2k)
out_members = group_members(out_tables)
out_patterns = {r["LoadPat"] for r in out_tables["LOAD PATTERN DEFINITIONS"]}
out_loads = {(r["Joint"], r["LoadPat"]): r for r in out_tables["JOINT LOADS - FORCE"]}
out_cases = {r["Case"] for r in out_tables["LOAD CASE DEFINITIONS"]}
out_assigned = {(r["Case"], r["LoadName"]) for r in out_tables["CASE - STATIC 1 - LOAD ASSIGNMENTS"]}

for node, joint in mapping.items():
    assert out_members.get(node) == {joint}, f"el grupo {node} cambió de joints"
    assert node in out_patterns, f"falta el estado {node}"
    assert node in out_cases, f"falta el caso de carga {node}"
    assert (node, node) in out_assigned, f"el caso {node} no tiene el estado asignado"
    row = out_loads.get((joint, node))
    assert row is not None, f"falta la carga en joint {joint}, estado {node}"
    got = [float(row[k]) for k in ("F1", "F2", "F3")]
    assert all(abs(g - e) < 1e-9 for g, e in zip(got, FORCE_KGF)), f"carga incorrecta en {node}: {got}"
print(f"Verificación del .s2k OK: {len(mapping)} nodos")

In [ ]:
if desp_path is None:
    print("Etapa 2 omitida: adjunta el Excel de Joint Displacements de SAP y vuelve a ejecutar.")
else:
    if desp_path.lower().endswith(".csv"):
        desp_raw = pd.read_csv(desp_path, dtype=str)
    else:
        desp_raw = pd.read_excel(desp_path, dtype=str)
    # SAP exporta encabezados de dos líneas ("Joint" / "Text", "U1" / "m"): se toma la primera palabra de cada uno.
    desp_raw.columns = [str(c).split()[0] if str(c).split() else str(c) for c in desp_raw.columns]
    # La fila de unidades que SAP deja bajo la cabecera no tiene valores numéricos en U1: se descarta abajo.
    desp = desp_raw[desp_raw["Joint"].notna() & desp_raw["OutputCase"].notna()].copy()
    desp = desp[pd.to_numeric(desp["U1"], errors="coerce").notna()].copy()
    for c in ["U1", "U2", "U3"]:
        desp[c] = pd.to_numeric(desp[c])
    # Excel puede guardar el joint o el caso como número ("1.0"): se normalizan igual que en el mapeo.
    desp["Joint"] = desp["Joint"].map(normalize_name)
    desp["OutputCase"] = desp["OutputCase"].map(normalize_name)

    dup = desp[desp.duplicated(["Joint", "OutputCase"], keep=False)]
    if len(dup):
        raise ValueError(f"Filas duplicadas (joint, caso):\n{dup.to_string()}")

    rows, faltan = [], []
    for node, joint in mapping.items():
        hit = desp[(desp["Joint"] == joint) & (desp["OutputCase"] == node)]
        if hit.empty:
            faltan.append(f"{node} (joint {joint})")
            continue
        u = hit.iloc[0][["U1", "U2", "U3"]].to_numpy(dtype=float)
        k = [np.inf if abs(ui) == 0 else F / ui for F, ui in zip(FORCE_KGF, u)]
        rows.append({"nodo_piping": node, "joint_sap": joint,
                     "U1": u[0], "U2": u[1], "U3": u[2],
                     "k_x": k[0], "k_y": k[1], "k_z": k[2]})

    if faltan:
        print("Sin resultado en el reporte de SAP (se listan, no se calculan):")
        print("  " + ", ".join(faltan))
    if not rows:
        raise ValueError("Ningún nodo tiene desplazamientos. El caso de salida debe llamarse igual que el nodo.")

    rep = pd.DataFrame(rows)
    rep.to_csv(out_dir / "rigidez_por_nodo.csv", index=False)
    rep.to_excel(out_dir / "rigidez_por_nodo.xlsx", index=False)
    print(f"Nodos con rigidez calculada: {len(rep)}. Escritos: rigidez_por_nodo.csv y .xlsx")
    rep